# Blood Work Analysis Pipeline
A two-stage LLM pipeline:
- **Stage 1:** Extract and flag abnormal values from a blood report
- **Stage 2:** Generate a health summary and Indian diet plan based on flagged values

In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

## Sample Blood Report

In [5]:
with open("blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5–17.5)
Hematocrit:        44%            


## Stage 1 — Extract and Flag Abnormal Values

In [6]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
Here are the extracted test values from the blood report:

**COMPLETE BLOOD COUNT (CBC)**
- Test Name: Hemoglobin | Status: NORMAL | Reference: 13.5–17.5 g/dL
- Test Name: Hematocrit | Status: NORMAL | Reference: 41–53%
- Test Name: WBC | Status: NORMAL | Reference: 4.5–11.0 x10^3/uL
- Test Name: Platelets | Status: NORMAL | Reference: 150–400 x10^3/uL

**LIPID PANEL**
- Test Name: Total Cholesterol | Status: HIGH | Reference: <200 mg/dL
- Test Name: LDL Cholesterol | Status: HIGH | Reference: <100 mg/dL
- Test Name: HDL Cholesterol | Status: LOW | Reference: >40 mg/dL
- Test Name: Triglycerides | Status: HIGH | Reference: <150 mg/dL

**METABOLIC PANEL**
- Test Name: Glucose (Fasting) | Status: NORMAL | Reference: 70–99 mg/dL
- Test Name: HbA1c | Status: NORMAL | Reference: <5.7%
- Test Name: Creatinine | Status: NORMAL | Reference: 0.7–1.3 mg/dL
- Test Name: eGFR | Status: NORMAL | Reference: >60 mL/min

**LIVER FUNCTION**
- Test Name: ALT | Status: N

## Stage 2 — Health Summary and Indian Diet Plan

In [7]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
### Health Summary
Your blood report indicates a condition called dyslipidemia, characterized by high "bad" cholesterol (LDL) and triglycerides, alongside low "good" cholesterol (HDL). While your blood sugar, kidney, and liver functions are currently healthy, your elevated lipid profile increases the risk of heart-related issues if left unmanaged. The good news is that these levels are highly responsive to dietary changes. By focusing on heart-healthy fats and reducing refined carbohydrates, you can significantly improve your cholesterol balance and overall cardiovascular health.

***

### Indian Diet Plan

**(1) Foods to avoid**
*   **Refined Carbohydrates:** White bread, maida (all-purpose flour) products like naan, biscuits, rusks, and polished white rice.
*   **Saturated and Trans Fats:** Dalda (vanaspati), ghee in excess, butter, deep-fried snacks (pakoras, samosas, mathris), and commercially processed bakery items.
*   **Sugary Additive